In [5]:
##List of disease and Orphacode from Orphanet database from xml file version 01-07-24 (https://www.orphadata.com/genes/)
##Try with ordo id from disgenet
##mail disgnet - get list of ordo annotated disease

##try with list of genes from orpahnet itself - first try this

import xml.etree.ElementTree as ET
import pandas as pd

with open("orphanet-list.xml", "rb") as f:
    tree = ET.parse(f)

root = tree.getroot()

data = []

for disorder in root.findall(".//Disorder"):
    orpha_code = disorder.findtext("OrphaCode")
    disease_name = disorder.findtext("Name")

    if orpha_code and disease_name:
        data.append((orpha_code.strip(), disease_name.strip()))

df = pd.DataFrame(data, columns=["OrphaCode", "Disease_Name"])

print(df.head())
print(f"Rows: {len(df)}")

df.to_csv("orphanet_disease_orphacode.csv", index=False)  

  OrphaCode                                       Disease_Name
0    166024  Multiple epiphyseal dysplasia-macrocephaly-fac...
1        93                             Aspartylglucosaminuria
2    166035  Brachydactyly-short stature-retinitis pigmento...
3       585                      Multiple sulfatase deficiency
4       118                                  Beta-mannosidosis
Rows: 3961


In [10]:
import xml.etree.ElementTree as ET
import pandas as pd

with open("orphanet-list.xml", "rb") as f:
    tree = ET.parse(f)

root = tree.getroot()

disease_info = []

for disorder in root.findall(".//Disorder"):
    name_tag = disorder.find("Name")
    code_tag = disorder.find("OrphaCode")
    disorder_id = disorder.get("id", "N/A")

    if name_tag is None or code_tag is None:
        continue

    disorder_name = name_tag.text.strip() if name_tag.text else "N/A"
    orpha_code = code_tag.text.strip() if code_tag.text else "N/A"

    genes = disorder.findall(".//Gene")

    if not genes:
        disease_info.append({
            "Disorder Name": disorder_name,
            "OrphaCode": orpha_code,
            "Disorder ID": disorder_id,
            "Gene Name": "N/A",
            "Gene Symbol": "N/A",
            "Ensembl Reference": "N/A"
        })
        continue

    for gene in genes:
        gene_name_tag = gene.find("Name")
        gene_symbol_tag = gene.find("Symbol")

        gene_name = gene_name_tag.text.strip() if gene_name_tag is not None and gene_name_tag.text else "N/A"
        gene_symbol = gene_symbol_tag.text.strip() if gene_symbol_tag is not None and gene_symbol_tag.text else "N/A"

        ensembl_ref = "N/A"

        for ext_ref in gene.findall(".//ExternalReference"):
            source_tag = ext_ref.find("Source")
            reference_tag = ext_ref.find("Reference")

            source_text = source_tag.text.strip() if source_tag is not None and source_tag.text else ""
            reference_text = reference_tag.text.strip() if reference_tag is not None and reference_tag.text else "N/A"

            if source_text == "Ensembl":
                ensembl_ref = reference_text
                break

        disease_info.append({
            "Disorder Name": disorder_name,
            "OrphaCode": orpha_code,
            "Disorder ID": disorder_id,
            "Gene Name": gene_name,
            "Gene Symbol": gene_symbol,
            "Ensembl Reference": ensembl_ref
        })

df = pd.DataFrame(disease_info)
df.to_csv("orphanet_gene_disease.csv", index=False)  

print(df.head())
print(df.shape)

                                       Disorder Name OrphaCode Disorder ID  \
0  Multiple epiphyseal dysplasia-macrocephaly-fac...    166024       17601   
1                             Aspartylglucosaminuria        93           5   
2  Brachydactyly-short stature-retinitis pigmento...    166035       17604   
3                      Multiple sulfatase deficiency       585           6   
4                                  Beta-mannosidosis       118           7   

                                  Gene Name Gene Symbol Ensembl Reference  
0                   kinesin family member 7        KIF7   ENSG00000166813  
1                   aspartylglucosaminidase         AGA   ENSG00000038002  
2  CWC27 spliceosome associated cyclophilin       CWC27   ENSG00000153015  
3              sulfatase modifying factor 1       SUMF1   ENSG00000144455  
4                          mannosidase beta       MANBA   ENSG00000109323  
(8164, 6)


In [15]:
import py4cytoscape as p4c
import IPython

print(f'Loading Javascript client ... {p4c.get_browser_client_channel()} on {p4c.get_jupyter_bridge_url()}')
browser_client_js = p4c.get_browser_client_js()
IPython.display.Javascript(browser_client_js)

Loading Javascript client ... 19980f42-4f46-40f1-a190-67df8a1329e9 on https://jupyter-bridge.cytoscape.org


<IPython.core.display.Javascript object>

In [16]:
import py4cytoscape as p4c

print(p4c.cytoscape_version_info())
p4c.cytoscape_ping()

{'apiVersion': 'v1', 'cytoscapeVersion': '3.10.3', 'automationAPIVersion': '1.13.0', 'py4cytoscapeVersion': '1.13.0', 'jupyterBridgeVersion': '0.0.2'}
You are connected to Cytoscape!


'You are connected to Cytoscape!'

In [17]:
import pandas as pd
import py4cytoscape as p4c

edges = pd.DataFrame({
    "source": ["A", "B", "C"],
    "target": ["B", "C", "D"]
})

p4c.create_network_from_data_frames(edges=edges, title="test_network")

Applying default style...
Applying preferred layout


131

In [21]:
import pandas as pd
import numpy as np

import pandas as pd
import py4cytoscape as p4c

import os
print('Get current working directory : ', os.getcwd())

from IPython.display import Image
from IPython.display import SVG

pd.set_option('display.max_colwidth', 1)


# Extract unique IDs for genes and KEs
unique_orpha_id = pd.unique(df[['OrphaCode']].values.ravel())
unique_gene_id = pd.unique(df[['Gene Symbol']].values.ravel())

# Combine the unique IDs
combined_ids = np.concatenate([unique_orpha_id, unique_gene_id])

nodes = pd.DataFrame({'id': combined_ids})
    # Create edges dataframe
edges = pd.DataFrame(data={'source': df['OrphaCode'], 'target': df['Gene Symbol'], 'interaction': 'interacts', 'weight': 1 })
print(nodes)  
    # Create network
p4c.create_network_from_data_frames(nodes, edges, title= "orphanet network")
disease_table = df[['OrphaCode', 'Disorder Name', 'Disorder ID']].drop_duplicates()
disease_table = disease_table.rename(columns={'OrphaCode': 'id'})

disease_table['group'] = np.where(
    disease_table['Disorder Name'].notna() & (disease_table['Disorder Name'].astype(str).str.strip() != ''),
    'disease',
    ''
)

p4c.load_table_data(
    data=disease_table,
    data_key_column='id',
    table='node',
    table_key_column='id'
)

Get current working directory :  /home/jovyan/work/persistent/orphanet
           id
0     166024 
1     93     
2     166035 
3     585    
4     118    
...   ...    
8416  KCNJ1  
8417  MIR140 
8418  NCKAP1L
8419  SOCS1  
8420  NFE2L2 

[8421 rows x 1 columns]
Applying default style...
Applying preferred layout


'Success: Data loaded in defaultnode table'

In [ ]:
##importing ppi from IntAct (try with string first- try physical network)

import wget

url = 'https://ftp.ebi.ac.uk/pub/databases/intact/2024-11-29/psimitab/species/human.txt'

filename = wget.download(url)

filename

KeyboardInterrupt: 

In [ ]:
from git import Repo

repo_url = "https://github.com/eugenebang/DREAMwalk.git"

repo = Repo.clone_from(
    repo_url,
    '/Users/aishw/Documents/project3/git/'    
)

In [29]:
conda

ValueError: The python kernel does not appear to be a conda environment.  Please use ``%pip install`` instead.